# Model Selection for Time-Respecting Paths in Temporal Graphs

## Prerequisites

First, we need to set up our Python environment that has PyTorch, PyTorch Geometric and PathpyG installed. Depending on where you are executing this notebook, this might already be (partially) done. E.g. Google Colab has PyTorch installed by default so we only need to install the remaining dependencies. The DevContainer that is part of our GitHub Repository on the other hand already has all of the necessary dependencies installed.

In the following, we install the packages for usage in Google Colab using Jupyter magic commands. For other environments comment in or out the commands as necessary. For more details on how to install `pathpyG` especially if you want to install it with GPU-support, we refer to our [documentation](https://www.pathpy.net/dev/getting_started/). Note that `%%capture` discards the full output of the cell to not clutter this tutorial with unnecessary installation details. If you want to print the output, you can comment `%%capture` out.

In [1]:
%%capture
# !pip install torch
# !pip install torch_geometric
# !pip install git+https://github.com/pathpy/pathpyG.git

## Motivation and Learning Objectives

In the tutorial on [path data and higher-order models](https://www.pathpy.net/dev/tutorial/paths_higher_order/) we saw that the same set of paths can be modelled by De Bruijn graphs of different orders $k$, which raises the question of the **optimal order**. There we answered it with a likelihood ratio test via `MultiOrderModel.estimate_order`, applied to a `PathData` object containing walks of varying length.

In the tutorial on [higher-order models for time-respecting paths](https://www.pathpy.net/dev/tutorial/trp_higher_order/) we saw that time-respecting paths in a temporal graph give rise to the same kind of higher-order De Bruijn graph models, computed efficiently from the temporal event graph without enumerating paths.

This tutorial connects the two: **how do we select the optimal order for time-respecting paths in a temporal graph?** We will see that this is not simply a matter of calling `estimate_order` on a model built with `from_temporal_graph`, because a likelihood ratio test imposes a requirement that such a model does not satisfy. In the following you will learn:

- why order selection needs an observation set that does not depend on the order being tested,
- how to use `MultiOrderModel.from_time_respecting_walks` to obtain one,
- what the `walk_length` parameter means and how it trades statistical power against the length of the temporal correlations you can detect,
- how to apply this to an empirical temporal graph.

In [2]:
import os
import tempfile
from urllib import request

import torch

import pathpyG as pp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## A Toy Example with Second-Order Structure

Let us construct a small temporal graph in which the causal structure is known by design. Node `b` acts as a hub. Interactions come in pairs: someone interacts with `b`, and immediately afterwards `b` interacts with someone else. We separate consecutive pairs by a gap in time, so that with $\delta=1$ only the two edges within a pair form a time-respecting path.

In the first graph, the two "channels" through the hub never mix: an interaction coming from `a` is always followed by an interaction to `c`, and one coming from `x` is always followed by one to `d`.

In [3]:
def hub_temporal_graph(pairs, repeats):
    # Build a temporal graph of two-event blocks first -> b -> second, separated in time.
    tedges = []
    for i, (first, second) in enumerate(list(pairs) * repeats):
        t = 3 * i  # a gap of two time units keeps consecutive blocks causally independent at delta=1
        tedges += [(first, "b", t), ("b", second, t + 1)]
    return pp.TemporalGraph.from_edge_list(tedges)


t_correlated = hub_temporal_graph([("a", "c"), ("x", "d")], repeats=12)
print(t_correlated)

Temporal Graph with 5 nodes, 4 unique edges and 48 events in [0, 70]
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


## What Exactly Are We Fitting the Model To?

Before we can test which order best describes this temporal graph, we have to be precise about **what the observed data is**. For `PathData` this was obvious: the data is the collection of walks we put in. For a temporal graph it is not, because time-respecting paths overlap and share events.

`pathpyG` resolves this by taking the observation set to be **all time-respecting walks consisting of exactly `walk_length` events**. We can make this concrete with `extract_time_respecting_walks`, which enumerates that set explicitly:

In [4]:
walks = pp.algorithms.extract_time_respecting_walks(t_correlated, delta=1, length=2)
print(f"{walks.num_paths} observed walks of two events")
print("distinct walks:", sorted(set(walks.get_walk(i) for i in range(walks.num_paths))))

24 observed walks of two events
distinct walks: [('a', 'b', 'c'), ('x', 'b', 'd')]


Only two of the four conceivable walks through the hub actually occur, so a first-order model - which would treat the continuation after `b` as independent of what came before - cannot reproduce this data.

<div class="admonition warning">
    <p class="admonition-title">Enumeration does not scale</p>
    <p>
        <code>extract_time_respecting_walks</code> materializes every walk and is meant for small examples and for validating results. The number of walks grows exponentially in the branching factor of the temporal graph. The model construction shown below never enumerates a single walk; it derives the same statistics directly from the temporal event graph.
    </p>
</div>

## Fitting a Multi-Order Model

To fit a multi-order model to this observation set we use `MultiOrderModel.from_time_respecting_walks`. Like `from_temporal_graph` it takes the maximum time difference $\delta$ and a maximum order, and it computes the layers from the temporal event graph.

In [5]:
m = pp.MultiOrderModel.from_time_respecting_walks(t_correlated, delta=1, max_order=2)
print(m)
print(m.layers[1])
print(m.layers[2])

MultiOrderModel with max. order 2
Directed graph with 5 nodes and 4 edges
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([4])"},
    'Graph Attributes': {'num_nodes': "<class 'int'>"},
    'Node Attributes': {   'node_instance_weight': "<class 'torch.Tensor'> -> torch.Size([5])",
                           'node_path_start_weight': "<class 'torch.Tensor'> -> torch.Size([5])"}}
Directed graph with 4 nodes and 2 edges
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([2])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([48])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {'node_path_start_weight': "<class 'torch.Tensor'> -> torch.Size([4])"}}


Each layer additionally carries the statistics needed to evaluate the likelihood of the observed walks. `node_path_start_weight` records, for every $k$-th order node, how many observed walks *begin* with it, so summing it gives back the number of observations - the same number at every order, which is exactly the property we will need in a moment.

In [6]:
for k in sorted(m.layers):
    print(f"order {k}: total path start weight = {m.layers[k].data.node_path_start_weight.sum().item():.0f}")

order 1: total path start weight = 24
order 2: total path start weight = 24


## Estimating the Optimal Order

We can now run the likelihood ratio test. Note that no path data has to be passed in: the model already carries everything the test needs.

In [7]:
print("estimated optimal order:", m.estimate_order(max_order=2))

estimated optimal order: 2


/workspaces/pathpyG/.venv/lib/python3.13/site-packages/torch_geometric/edge_index.py:863: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  return torch.sparse_csr_tensor(


As expected, a second-order model is required. Let us contrast this with a temporal graph built from the same hub topology, but in which all four combinations occur equally often. Here the continuation after `b` carries no information about what preceded it, so a first-order model suffices:

In [8]:
t_independent = hub_temporal_graph([("a", "c"), ("a", "d"), ("x", "c"), ("x", "d")], repeats=3)

walks_ind = pp.algorithms.extract_time_respecting_walks(t_independent, delta=1, length=2)
print("distinct walks:", sorted(set(walks_ind.get_walk(i) for i in range(walks_ind.num_paths))))

m_ind = pp.MultiOrderModel.from_time_respecting_walks(t_independent, delta=1, max_order=2)
print("estimated optimal order:", m_ind.estimate_order(max_order=2))

distinct walks: [('a', 'b', 'c'), ('a', 'b', 'd'), ('x', 'b', 'c'), ('x', 'b', 'd')]
estimated optimal order: 1


## Why Not `from_temporal_graph`?

It is natural to ask why we need a separate constructor at all, rather than calling `estimate_order` on a model built with `from_temporal_graph`. Trying it produces an error:

In [9]:
m_legacy = pp.MultiOrderModel.from_temporal_graph(t_correlated, delta=1, max_order=2)
try:
    m_legacy.estimate_order(max_order=2)
except ValueError as e:
    print(e)

2026-07-31 11:11:24 - Layer of order 1 does not carry the statistic 'node_instance_weight'.


Layer of order 1 does not carry 'node_instance_weight', so its likelihood cannot be computed. Likelihoods need a fixed observation set: build the model with `MultiOrderModel.from_path_data` or `MultiOrderModel.from_time_respecting_walks`. `from_temporal_graph` weights each layer by the number of distinct walks of that length, which makes the observed data depend on the order being tested.


The reason is statistical rather than technical. A likelihood ratio test compares two hypotheses by the ratio of their likelihoods, which is only meaningful if both are likelihoods of the **same data**.

`from_temporal_graph` weights an edge in layer $k$ by the number of *distinct* time-respecting walks of $k$ events. That is exactly the right quantity for describing the causal topology of a temporal graph, which is what that constructor is for, and it is what the higher-order models used elsewhere in `pathpyG` are built on. But it makes the observed data a function of $k$: a model of order 2 would be describing the set of two-event walks, and a model of order 3 the set of three-event walks. Comparing their likelihoods would compare two different datasets and produce a number that looks like a p-value but tests nothing.

`from_time_respecting_walks` fixes the observation set to all walks of exactly `walk_length` events, chosen independently of the order under test. Every order is then evaluated against the same data and the test is valid. This is why the two constructors coexist: they answer different questions, and `from_temporal_graph` keeps its original meaning.

## Choosing `walk_length`

By default `walk_length` equals `max_order`, but it can be set explicitly. It has a direct interpretation: it is the length of the observations, and therefore determines both how much data the test sees and how long the temporal correlations are that the data can possibly reveal.

Longer observations are more informative individually, but there are fewer of them, because a walk must survive to the full length to be counted at all. The following shows the effect on our toy example, where nothing extends beyond two events:

In [10]:
for length in [1, 2, 3]:
    n = pp.algorithms.extract_time_respecting_walks(t_correlated, delta=1, length=length).num_paths
    print(f"walk_length={length}: {n} observed walks")

walk_length=1: 48 observed walks
walk_length=2: 24 observed walks


walk_length=3: 0 observed walks


<div class="admonition warning">
    <p class="admonition-title">Models with different <code>walk_length</code> are not comparable</p>
    <p>
        A model's <code>walk_length</code> defines its observation set. Two models built with different values describe different data, so their likelihoods - and any test statistic derived from them - cannot be compared. Since <code>walk_length</code> defaults to <code>max_order</code>, this also means that raising <code>max_order</code> silently changes the data unless you pin <code>walk_length</code> yourself. Within a single <code>estimate_order</code> call the observation set is always fixed, which is what makes that call valid.
    </p>
</div>

## An Empirical Temporal Graph

Let us apply this to a larger example: a synthetic temporal graph with 60,000 time-stamped interactions between 30 nodes, which contains temporal-topological cluster structure. It is the same dataset used in the tutorial on [causality-aware graph neural networks](https://www.pathpy.net/dev/tutorial/dbgnn/).

<div class="admonition note">
    <p class="admonition-title">Dataset Availability</p>
    <p>
        Depending on how you are executing this notebook, you may need to download the dataset first. We automatically check if the dataset is available in the relative path <code>../data/temporal_clusters.tedges</code>, which is the default location if you cloned the pathpyG repository. If the file is not found, we download it from the GitHub repository.
    </p>
</div>

In [11]:
if os.path.exists('../data/temporal_clusters.tedges'):
    print("Loading dataset from local path...")
    t_emp = pp.io.read_csv_temporal_graph('../data/temporal_clusters.tedges', header=False)
else:
    print("Loading dataset from remote URL...")
    with tempfile.TemporaryDirectory() as tmpdir:
        url = "https://raw.githubusercontent.com/pathpy/pathpyG/refs/heads/main/docs/data/temporal_clusters.tedges"
        file_path = os.path.join(tmpdir, 'temporal_clusters.tedges')
        request.urlretrieve(url, file_path)
        t_emp = pp.io.read_csv_temporal_graph(file_path, header=False)

print(t_emp)

Loading dataset from local path...


Temporal Graph with 30 nodes, 557 unique edges and 60000 events in [0, 59999]
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


Constructing the temporal event graph is the expensive part of the computation, so we compute it once per $\delta$ and pass it to the model constructor via the `event_graph` argument. This lets us vary `walk_length` cheaply.

We compare a small and a large maximum time difference. Since every one of the 60,000 events has its own time stamp, $\delta=1$ admits only interactions in immediately consecutive time steps, while $\delta=20$ admits considerably more.

In [12]:
%%capture
# %%capture suppresses the progress bars of the two event graph constructions.
results = {}
for delta in [1, 20]:
    event_graph = pp.algorithms.lift_order_temporal(t_emp.to(device), delta=delta)
    for length in [2, 3, 4]:
        model = pp.MultiOrderModel.from_time_respecting_walks(
            t_emp, delta=delta, max_order=length, walk_length=length, event_graph=event_graph
        )
        results[(delta, length)] = (
            model.layers[1].data.node_path_start_weight.sum().item(),
            model.estimate_order(max_order=length),
        )

In [13]:
print(f"{'delta':>6} {'walk_length':>12} {'observations':>13} {'optimal order':>14}")
for (delta, length), (num_obs, order) in results.items():
    print(f"{delta:>6} {length:>12} {num_obs:>13,.0f} {order:>14}")

 delta  walk_length  observations  optimal order
     1            2        30,968              2
     1            3         1,936              1
     1            4         1,005              1
    20            2        69,088              2
    20            3        65,439              2
    20            4        65,958              3


Two things stand out.

First, the dataset does contain genuine higher-order structure: at $\delta=20$ a first-order model is rejected at every observation length, and at `walk_length=4` even a second-order model is not sufficient. This is what we would hope for in data that was generated with temporal-topological clusters.

Second, and more instructive, the result at $\delta=1$ depends strongly on `walk_length`. With two-event observations there are around 31,000 of them and the test comfortably rejects a first-order model. With four-event observations only about a thousand walks survive, and the test can no longer reject it. This is not a failure of the method but its correct behaviour: the longer observation set is genuinely much smaller, and a likelihood ratio test that sees little data has little power. If you find that raising the order lowers the detected order, check the number of observations before concluding that there is no higher-order structure - you may simply have starved the test.

Increasing $\delta$ is the remedy here, because it allows more events to form time-respecting paths and keeps the number of long observations high.

## Summary

- Order selection for time-respecting paths requires an observation set that does not depend on the order being tested. `MultiOrderModel.from_time_respecting_walks` provides one by fixing the number of events per observation.
- `MultiOrderModel.from_temporal_graph` describes the causal topology of a temporal graph and remains the right tool for that purpose, but its layer weights make the observed data depend on the order, so it cannot be used for likelihood ratio tests.
- `estimate_order` needs no path data: the required statistics are attached to the model layers when it is built.
- `walk_length` and $\delta$ jointly control how much evidence the test sees. Always inspect the number of observations alongside the estimated order.